# SPLADE: side-experiment — v2 full train, v1 только eval

Отдельный блокнот рядом с основным `splade_experiments.ipynb`. Отличие одно:
**этап train для v1 исключён** — для v1 берётся уже обученная (последняя) модель
из `outputs/v1_splade_max/` и прогоняется только инференс/eval. v2 прогоняется
полностью (train + eval), как в основном блокноте.

Чтобы не трогать `splade_lab/` и не сломать поведение основного блокнота, вся
новая логика (eval-only без train) вынесена **прямо в этот блокнот** — это
side-experiment. Используются только готовые кирпичики из `splade_lab` (модель,
данные, eval, артефакты) без их изменения.

Порядок:
1. **Setup** — окружение и устройство.
2. **Конфиг данных** — `MODE = "full"` и источники MS MARCO.
3. **Конфиг эксперимента** — только v2 (v1 не обучается).
4. **Данные** — переиспользуются (скачаны основным блокнотом).
5. **v1 eval-only** — последняя модель из `outputs/v1_splade_max/` -> eval.
6. **v2 full** — обучение + eval.
7. **Сравнение** — сводная таблица метрик всех прогонов.

## 1. Setup

In [1]:
print("WORKS")

WORKS


In [2]:
# Первый запуск на новой машине (в активированном venv) — раскомментировать:
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

import json
import torch

from splade_lab.config import merge_config
from splade_lab.train import pick_device

device = pick_device()
print(f"torch {torch.__version__} | устройство: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

Looking in indexes: https://pypi.yandex-team.ru/simple/


Note: you may need to restart the kernel to use updated packages.


torch 2.5.1+cu124 | устройство: cuda (NVIDIA A100 80GB PCIe)


## 2. Конфиг данных

Те же данные, что и в основном блокноте (`full` — коллекция 8.8M пассажей + dev small).
Если основной блокнот уже подготовил `data/msmarco/full`, повторное скачивание не нужно.

In [3]:
MODE = "full"  # side-experiment по умолчанию в full (v1 берётся из готовых outputs)

DATA = {
    # Официальные файлы Microsoft. Если хост недоступен — зеркало:
    # https://msmarco.blob.core.windows.net/msmarcoranking/
    "urls": {
        "triples": "https://msmarco.z22.web.core.windows.net/msmarcoranking/triples.train.small.tar.gz",
        "collection": "https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz",
        "queries": "https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz",
        "qrels_dev": "https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.small.tsv",
    },
    "data_dir": "data/msmarco",  # относительно корня репозитория
    "smoke": {
        "num_train_triples": 256,  # строк triples на обучение
        "num_eval_queries": 20,    # уникальных eval-запросов (не пересекаются с train)
        "num_corpus_docs": 1000,   # корпус: positives eval-запросов + дистракторы
    },
    "full": {
        "num_train_triples": 3_200_000,  # ~ max_steps * batch_size; -1 = весь файл (39.8M строк)
        "num_eval_queries": -1,          # -1 = все 6980 запросов dev small
    },
}

print(f"MODE = {MODE} | параметры данных: {DATA[MODE]}")

MODE = full | параметры данных: {'num_train_triples': 3200000, 'num_eval_queries': -1}


## 3. Конфиг эксперимента (только v2)

Здесь обучается **только v2_splade_doc** (SPLADE-doc). v1 не обучается — его веса
берутся готовыми из `outputs/v1_splade_max/` на этапе eval (клетка 5).

`BASE`/`SMOKE_OVERRIDES`/`build_config` совпадают с основным блокнотом, чтобы
конфиг v2 был идентичен.

In [4]:
# База: гиперпараметры статьи (distilbert, Adam 2e-5, warmup 6000, max_len 256).
BASE = {
    "model": {
        "hf_model": "distilbert-base-uncased",
        "query_encoder": "mlm",      # mlm = SPLADE-max | bow = SPLADE-doc
        "max_len_query": 32,
        "max_len_doc": 256,
    },
    "train": {
        "seed": 4,
        "lr": 2e-5,
        "batch_size": 64,
        "max_steps": 100_000,
        "warmup_steps": 6_000,
        "flops_warmup_steps": 10_000,  # квадратичный разгон лямбд (SPLADE v2)
        "lambda_q": 3e-4,              # FLOPS-рег. запросов
        "lambda_d": 1e-4,              # FLOPS-рег. документов
        "log_every": 100,
    },
    "eval": {
        "batch_size_docs": 256,
        "batch_size_queries": 64,
        "batch_size_search": 64,
        "recall_ks": [10, 100, 1000],
    },
}

# Smoke: всё маленькое, схема артефактов та же.
SMOKE_OVERRIDES = {
    "model": {"max_len_doc": 128},
    "train": {"max_steps": 20, "batch_size": 8, "warmup_steps": 4,
              "flops_warmup_steps": 8, "log_every": 5},
    "eval": {"batch_size_docs": 32, "batch_size_queries": 32, "recall_ks": [10, 100]},
}

# В этом блокноте обучаем только v2 (v1 — eval-only из готовых весов).
EXPERIMENTS = {
    "v2_splade_doc": {
        "model": {"query_encoder": "bow"},
        "train": {
                    "lambda_q": 0.0,      # запрос фиксирован, рег. не нужна
                    "lambda_d": 3e-4,     # рег. только на документы
                    "max_steps": 100_000
                },
    },
}

def build_config(name: str) -> dict:
    cfg = merge_config(BASE, EXPERIMENTS[name])
    if MODE == "smoke":
        cfg = merge_config(cfg, SMOKE_OVERRIDES)
    cfg["version"], cfg["mode"] = name, MODE
    return cfg

for name in EXPERIMENTS:
    print(f"=== {name} (mode={MODE}) ===")
    print(json.dumps(build_config(name), indent=2, ensure_ascii=False))

=== v2_splade_doc (mode=full) ===
{
  "model": {
    "hf_model": "distilbert-base-uncased",
    "query_encoder": "bow",
    "max_len_query": 32,
    "max_len_doc": 256
  },
  "train": {
    "seed": 4,
    "lr": 2e-05,
    "batch_size": 64,
    "max_steps": 100000,
    "warmup_steps": 6000,
    "flops_warmup_steps": 10000,
    "lambda_q": 0.0,
    "lambda_d": 0.0003,
    "log_every": 100
  },
  "eval": {
    "batch_size_docs": 256,
    "batch_size_queries": 64,
    "batch_size_search": 64,
    "recall_ks": [
      10,
      100,
      1000
    ]
  },
  "version": "v2_splade_doc",
  "mode": "full"
}


## 4. Данные (переиспользуются)

In [5]:
from splade_lab import data as msdata

prepare = msdata.prepare_full if MODE == "full" else msdata.prepare_smoke
ds_dir = prepare(DATA)

print(f"\nКаталог: {ds_dir}")
print("Строк в файлах:", msdata.dataset_stats(ds_dir))

[data] full уже готов: /home/uvuv/workspace/sandbox_03/data/msmarco/full

Каталог: /home/uvuv/workspace/sandbox_03/data/msmarco/full


Строк в файлах: {'collection.tsv': 8841823, 'queries.tsv': 6980, 'qrels.tsv': 7437, 'triples.tsv': 3200000}


## 5. v1 eval-only (инференс последней модели из outputs)

Код инлайн — `splade_lab/` не меняем, поведение основного блокнота не затрагиваем.

`find_latest_model_run` ищет в `outputs/<version>/` самый свежий прогон с сохранённой
моделью (`run_id` = timestamp, поэтому максимум по имени = последний). `evaluate_only`
грузит эти веса/токенайзер локально (без обращения к huggingface.co), прогоняет
`evaluate.evaluate_retrieval` и пишет артефакты как обычный прогон (config/metrics/meta
+ `source_run`), чтобы строка попала в сводную таблицу `compare_runs()`.

In [6]:
# --- side-experiment: eval-only без train (код инлайн, splade_lab не меняем) ---
from transformers import AutoTokenizer

from splade_lab import artifacts, data, evaluate
from splade_lab.config import OUTPUTS_DIR
from splade_lab.model import Splade


def find_latest_model_run(version: str):
    """Последний прогон версии в outputs/<version>/ с сохранённой моделью (model/)."""
    base = OUTPUTS_DIR / version
    runs = [d for d in sorted(base.glob("*"))
            if d.is_dir() and (d / "model" / "config.json").exists()]
    if not runs:
        raise FileNotFoundError(f"Нет прогонов с сохранённой моделью в {base}")
    return runs[-1]  # run_id = timestamp YYYYMMDD-HHMMSS -> последний по имени = свежайший


def evaluate_only(version: str, data_cfg: dict, mode: str, recall_ks=None):
    """Только eval: берём последнюю модель из outputs/<version>/, train пропускаем.

    Артефакты пишутся как у обычного прогона (config/metrics/meta + ссылка на
    source_run), поэтому результат попадает в сводную таблицу compare_runs().
    """
    src_run = find_latest_model_run(version)
    model_dir = src_run / "model"

    # Конфиг исходной модели (для max_len/query_encoder/recall_ks).
    cfg = json.loads((src_run / "config.json").read_text(encoding="utf-8"))
    cfg["version"], cfg["mode"] = version, mode
    if recall_ks is not None:
        cfg["eval"]["recall_ks"] = recall_ks

    ds_dir = data.dataset_dir(data_cfg, mode)
    if not data.is_prepared(data_cfg, mode):
        raise RuntimeError(f"Данные не готовы: {ds_dir} "
                           f"(запустите клетку подготовки данных, mode={mode}).")

    device = pick_device()
    run_dir = artifacts.create_run_dir(version)
    artifacts.save_config(run_dir, cfg)
    meta = artifacts.start_meta(run_dir, cfg, device)
    meta["eval_only"] = True
    meta["source_run"] = src_run.name
    print(f"[eval-only] {version} mode={mode} device={device}")
    print(f"[eval-only] модель: {model_dir} -> {run_dir}")

    # Локальная загрузка обученных весов (from_pretrained по пути -> сети не нужно).
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    model = Splade(str(model_dir), cfg["model"]["query_encoder"]).to(device)

    metrics = evaluate.evaluate_retrieval(model, tokenizer, ds_dir, cfg, device)
    metrics["final_train_loss"] = None                   # train не запускался
    metrics["train_steps"] = cfg["train"]["max_steps"]   # шаги исходной модели (справочно)
    metrics["source_run"] = src_run.name
    artifacts.save_metrics(run_dir, metrics)
    artifacts.finish_meta(run_dir, meta)
    print(f"[eval-only] готово: {run_dir}")
    print(f"[eval-only] метрики: {metrics}")
    return run_dir, metrics

In [7]:
results = {}

# # v1 — БЕЗ train: только инференс последней модели из outputs/v1_splade_max/.
# run_dir, metrics = evaluate_only("v1_splade_max", DATA, MODE)
# results["v1_splade_max"] = {"run_dir": str(run_dir), **metrics}
# print(f"v1_splade_max (eval-only): mrr@10={metrics.get('mrr@10'):.4f} | {run_dir}")

## 6. v2 full (обучение + eval)

v2 прогоняется полностью через штатный `run_experiment` (train -> save -> eval),
как в основном блокноте.

In [8]:
from splade_lab.train import run_experiment

cfg = build_config("v2_splade_doc")
run_dir, metrics = run_experiment(cfg, DATA)
results["v2_splade_doc"] = {"run_dir": str(run_dir), **metrics}
print(f"v2_splade_doc (full): mrr@10={metrics.get('mrr@10'):.4f} | {run_dir}")

[run] v2_splade_doc mode=full device=cuda -> /home/uvuv/workspace/sandbox_03/outputs/v2_splade_doc/20260613-180612


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [Errno 13] Permission denied"))'), '(Request ID: 92848a7d-2ad2-48ab-8477-1b37747f9336)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Retrying in 1s [Retry 1/5].


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [Errno 13] Permission denied"))'), '(Request ID: 0a591798-3ac2-40e5-9fa6-79542759f6a2)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


Retrying in 2s [Retry 2/5].


...

Retrying in 8s [Retry 5/5].


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [Errno 13] Permission denied"))'), '(Request ID: 4c5d1677-ef5d-4a84-a453-f8c238bea0ac)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json


[progress] лог прогресса: /home/uvuv/workspace/sandbox_03/outputs/logs/progress_20260613-210827.log


/home/uvuv/workspace/sandbox_03/splade_lab/evaluate.py:73: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  D = torch.sparse_csr_tensor(


[run] готово: /home/uvuv/workspace/sandbox_03/outputs/v2_splade_doc/20260613-180612
[run] метрики: {'mrr@10': 0.3000889161777414, 'recall@10': 0.546525787965616, 'recall@100': 0.8010506208213946, 'recall@1000': 0.9263849092645655, 'n_eval_queries': 6980, 'n_corpus_docs': 8841823, 'avg_nnz_doc': 175.1565878439322, 'avg_nnz_query': 6.8878223495702, 'final_train_loss': 0.03997361958026886, 'train_steps': 100000}
v2_splade_doc (full): mrr@10=0.3001 | /home/uvuv/workspace/sandbox_03/outputs/v2_splade_doc/20260613-180612


## 7. Сравнение версий (все прогоны из outputs/)

In [9]:
from splade_lab.compare import compare_runs

df = compare_runs()  # печатает таблицу и пишет outputs/comparison.csv
df

      version          run_id  mode  seed   mrr@10  recall@10  recall@100  recall@1000  avg_nnz_doc  avg_nnz_query  final_train_loss  train_steps  n_eval_queries  n_corpus_docs  duration_s
v1_splade_max 20260612-003154  full     4 0.323141   0.588933    0.854740     0.964649   147.953626      28.976648          0.001052       250000            6980        8841823     45874.6
v1_splade_max 20260613-161423  full     4 0.326386   0.604071    0.866010     0.965926   198.071323      22.267049               NaN       100000            6980        8841823      4358.7
v2_splade_doc 20260613-180612  full     4 0.300089   0.546526    0.801051     0.926385   175.156588       6.887822          0.039974       100000            6980        8841823     19145.3
v1_splade_max 20260612-002533 smoke    42 0.248175   0.550000    0.850000          NaN  5150.361000    1145.800000         34.342426           20              20           1000        10.5
v1_splade_max 20260612-002635 smoke     4 0.289008   0.

,version,run_id,mode,seed,mrr@10,recall@10,recall@100,recall@1000,avg_nnz_doc,avg_nnz_query,final_train_loss,train_steps,n_eval_queries,n_corpus_docs,duration_s
0,v1_splade_max,20260612-003154,full,4,0.323141,0.588933,0.854740,0.964649,147.953626,28.976648,0.001052,250000,6980,8841823,45874.6
1,v1_splade_max,20260613-161423,full,4,0.326386,0.604071,0.866010,0.965926,198.071323,22.267049,NaN,100000,6980,8841823,4358.7
2,v2_splade_doc,20260613-180612,full,4,0.300089,0.546526,0.801051,0.926385,175.156588,6.887822,0.039974,100000,6980,8841823,19145.3
3,v1_splade_max,20260612-002533,smoke,42,0.248175,0.550000,0.850000,NaN,5150.361000,1145.800000,34.342426,20,20,1000,10.5
4,v1_splade_max,20260612-002635,smoke,4,0.289008,0.650000,0.900000,NaN,4406.685000,1073.750000,28.770599,20,20,1000,2.5
5,v2_splade_doc,20260612-002543,smoke,42,0.879167,1.000000,1.000000,NaN,985.329000,7.450000,0.525460,20,20,1000,2.2
6,v2_splade_doc,20260612-002637,smoke,4,0.912500,1.000000,1.000000,NaN,826.187000,7.450000,0.549977,20,20,1000,2.1


## Примечания

- Это **side-experiment**: вся новая логика (eval-only) живёт в клетке 5 этого
  блокнота, `splade_lab/` и основной блокнот не изменены.
- v1 НЕ обучается — `evaluate_only("v1_splade_max", ...)` берёт **последнюю** по
  timestamp модель из `outputs/v1_splade_max/` и делает только инференс/eval.
- Чтобы переоценить конкретный прогон v1 (не последний), временно убери лишние
  каталоги из `outputs/v1_splade_max/` или поправь `find_latest_model_run`.
- eval-only прогон пишется в новый `outputs/v1_splade_max/<run_id>/` с пометкой
  `eval_only` и ссылкой `source_run` в `meta.json`.